In [5]:
!pip install plotly nbformat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 31.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [nbformat]4/8 [plotly]s]


## Run

### loading the model

In [1]:
import cv2
import numpy as np
from notebook.utils import setup_sam_3d_body
from tools.vis_utils import visualize_sample_together

# Set up the estimator
estimator = setup_sam_3d_body(hf_repo_id="facebook/sam-3d-body-dinov3")

/workspace/sam-3d-body-measurement/sam_3d_body/models/heads/mhr_head.py:33: UserWarning: Momentum is not enabled
  warnings.warn("Momentum is not enabled")


Loading SAM 3D Body model from facebook/sam-3d-body-dinov3...


/root/miniconda3/envs/sam_3d_body/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:28<00:00,  5.78s/it]


Loading SAM 3D Body model...
Downloading: "https://github.com/facebookresearch/dinov3/zipball/main" to /root/.cache/torch/hub/main.zip


Ignored kwargs: {'drop_path': 0.1}
The model and loaded state dict do not match exactly

missing keys in source state_dict: backbone.encoder.mask_token, head_pose.hand_pose_comps_ori, head_pose.mhr.face_expressions_model.shape_vectors, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_indices, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_weight, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.2.weight, head_pose.mhr.character_torch.skeleton.joint_translation_offsets, head_pose.mhr.character_torch.skeleton.joint_prerotations, head_pose.mhr.character_torch.skeleton.pmi, head_pose.mhr.character_torch.skeleton.joint_parents, head_pose.mhr.character_torch.mesh.rest_vertices, head_pose.mhr.character_torch.mesh.faces, head_pose.mhr.character_torch.mesh.texcoords, head_pose.mhr.character_torch.mesh.texcoord_faces, head_pose.mhr.character_torch.parameter_transform.parameter_transform, head_pose.mhr.character_torch.parameter_transform.pose_parameters

Loading human detector from vitdet...
########### Using human detector: ViTDet...


/root/miniconda3/envs/sam_3d_body/lib/python3.11/site-packages/detectron2/config/lazy.py:167: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return old_import(name, globals, locals, fromlist=fromlist, level=level)
/root/miniconda3/envs/sam_3d_body/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
model_final_f05665.pkl: 2.77GB [01:10, 39.5MB/s]                               


Loading FOV estimator from moge2...
########### Using fov estimator: MoGe2...
Mask-condition inference is not supported...
Setup complete!
  Human detector: ✓
  Human segmentor: ✗ (mask inference disabled)
  FOV estimator: ✓


### inference

In [2]:
# Load and process image
img_bgr = cv2.imread("/workspace/sam-3d-body-measurement/notebook/images/p_s.png")
outputs = estimator.process_one_image(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))


####### Please make sure the input image is in RGB format
Running object detector...


/root/miniconda3/envs/sam_3d_body/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W1123 08:53:37.676000 3970 site-packages/torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


Found boxes: [[220.54501   18.123442 469.0481   977.8885  ]]
Running FOV estimator ...


### Experiments

#### measure

In [3]:
import torch
import trimesh
import numpy as np

def get_tpose_mesh_final(estimator, outputs):
    # --- 1. Locate the MHR Layer ---
    # Based on your previous logs:
    mhr_layer = estimator.model.head_pose.mhr
    device = next(mhr_layer.parameters()).device
    
    # --- 2. Prepare Tensors (Corrected Dimensions) ---
    
    # A. Identity (Shape)
    # Taken from inference output. Size: [1, 45]
    raw_shape = outputs[0]['shape_params']
    id_coeffs = torch.tensor(raw_shape).float().to(device).unsqueeze(0)
    
    # B. Pose (Model Parameters) - THE FIX
    # The error proved the model expects 249 total inputs.
    # 249 - 45 (Shape) = 204 (Pose).
    # We must provide 204 zeros for the T-Pose.
    pose_coeffs = torch.zeros((1, 204)).float().to(device)
    
    # C. Face Expression
    # Standard MHR size: 72
    face_coeffs = torch.zeros((1, 72)).float().to(device)

    # --- 3. Generate Mesh ---
    # We print shapes to confirm they sum to 249 (204+45)
    print(f"Generating T-Pose with: Shape={id_coeffs.shape[1]} + Pose={pose_coeffs.shape[1]} = {id_coeffs.shape[1]+pose_coeffs.shape[1]}")
    
    with torch.no_grad():
        # Positional arguments: identity, pose, face, correctives
        res = mhr_layer(
            id_coeffs,      
            pose_coeffs,    
            face_coeffs,    
            True            
        )

    # --- 4. Parse Output ---
    # Unpack the tuple ((verts, joints)) structure
    if isinstance(res, tuple):
        if isinstance(res[0], (tuple, list)):
            verts = res[0][0]
            joints = res[0][1]
        else:
            verts = res[0]
            joints = res[1]
    else:
        # Fallback for single return
        verts = res
        joints = None

    # --- 5. Get Faces ---
    # Fallback to standard MHR face logic if attribute is hidden
    try:
        if hasattr(mhr_layer, 'character_torch'):
            faces = mhr_layer.character_torch.mesh.faces
        else:
            # Try the path from your previous logs
            faces = estimator.model.head_pose.mhr.character_torch.mesh.faces
        faces_np = faces.cpu().numpy()
    except:
        print("Warning: Could not find faces array. Exporting Point Cloud only.")
        faces_np = None

    # --- 6. Return Trimesh ---
    verts_np = verts.cpu().numpy()[0]
    
    # Create the mesh
    mesh = trimesh.Trimesh(vertices=verts_np, faces=faces_np)
    
    # Return joints if available, otherwise we might need to recalculate them
    if joints is not None:
        joints_np = joints.cpu().numpy()[0]
    else:
        joints_np = None
        
    return mesh, joints_np

# # --- Execution ---
try:
    # 1. Run the generator
    tpose_mesh, tpose_joints = get_tpose_mesh_final(estimator, outputs)
    print(f"\nSUCCESS: Generated T-Pose Mesh.")
    print(f"Vertices: {len(tpose_mesh.vertices)}")
    print(f"Joints: {len(tpose_joints) if tpose_joints is not None else 0}")

except Exception as e:
    print(f"Error: {e}")

Generating T-Pose with: Shape=45 + Pose=204 = 249

SUCCESS: Generated T-Pose Mesh.
Vertices: 18439
Joints: 127


#### visualize

In [ ]:
import plotly.graph_objects as go
import numpy as np
import trimesh

# ==========================================
# 1. CONFIGURATION
# ==========================================

MHR_JOINT_MAP = {
    "pelvis": 1, "spine1": 37, "spine2": 36, "spine3": 35,
    "neck_base": 110, "r_shoulder": 69, "r_elbow": 73, "r_wrist": 60
}

MEASUREMENT_CONFIG = {
    # --- CIRCUMFERENCES ---
    "Chest": {
        "type": "circumference",
        "landmarks": ["spine1", "spine2"],
        "func": np.mean,
        "offset": 3.0,
        "color": "blue"
    },
    "Waist": {
        "type": "circumference",
        "landmarks": ["spine2", "spine3"],
        "func": np.mean,
        "offset": 0.0,
        "color": "green"
    },
    "Hips": {
        "type": "circumference",
        "landmarks": ["pelvis"],
        "func": np.mean,
        "offset": 0.0,
        "color": "purple"
    },
    
    # --- NEW: POLYLINE (Curved Measurements) ---
    "Spinal Length": {
        "type": "polyline", 
        # Define the path the tape measure should follow
        "landmarks": ["neck_base", "spine1", "spine2", "spine3", "pelvis"], 
        "color": "orange"
    },
    
    # --- LINEAR (Straight Lines) ---
    "Arm Length": {
        "type": "polyline", # Changed to polyline to include elbow bend
        "landmarks": ["r_shoulder", "r_elbow", "r_wrist"],
        "color": "cyan"
    }
}

# ==========================================
# 2. GEOMETRY ENGINE
# ==========================================

def scale_model(mesh, joints, target_height_cm):
    current_height = mesh.vertices[:, 1].max() - mesh.vertices[:, 1].min()
    scale_factor = target_height_cm / current_height
    mesh.vertices *= scale_factor
    joints *= scale_factor
    return mesh, joints, scale_factor

def get_largest_slice_loop(mesh, height_y):
    slice_obj = mesh.section(plane_origin=[0, height_y, 0], plane_normal=[0, 1, 0])
    if not slice_obj: return 0.0, None

    max_len = 0.0
    best_path = None
    for loop_points in slice_obj.discrete:
        diffs = np.diff(loop_points, axis=0)
        loop_len = np.sum(np.sqrt(np.sum(diffs**2, axis=1)))
        loop_len += np.linalg.norm(loop_points[-1] - loop_points[0])
        
        if loop_len > max_len:
            max_len = loop_len
            best_path = loop_points
    return max_len, best_path

# ==========================================
# 3. MEASUREMENT PROCESSOR (UPDATED)
# ==========================================

def calculate_body_metrics(mesh, joints, idx_map, config):
    results = {}

    for name, rule in config.items():
        color = rule.get("color", "white")

        # --- A. CIRCUMFERENCE ---
        if rule["type"] == "circumference":
            y_coords = [joints[idx_map[j]][1] for j in rule["landmarks"]]
            target_y = rule["func"](y_coords) + rule["offset"]
            val, path = get_largest_slice_loop(mesh, target_y)
            
            results[name] = {
                'type': 'circ', 'value': val, 'path': path, 'color': color
            }

        # --- B. POLYLINE (Chain of Segments) ---
        elif rule["type"] == "polyline":
            chain_indices = [idx_map[j] for j in rule["landmarks"]]
            chain_points = joints[chain_indices]
            
            # Calculate total length by summing segments
            total_dist = 0.0
            for i in range(len(chain_points) - 1):
                p1 = chain_points[i]
                p2 = chain_points[i+1]
                total_dist += np.linalg.norm(p1 - p2)

            results[name] = {
                'type': 'polyline', # We treat linear and polyline similarly in viz now
                'value': total_dist,
                'points': chain_points,
                'color': color
            }
            
        # --- C. LINEAR (Simple Start-End) ---
        elif rule["type"] == "linear":
            p_start = joints[idx_map[rule["start_joint"]]]
            p_end   = joints[idx_map[rule["end_joint"]]]
            dist = np.linalg.norm(p_start - p_end)
            
            results[name] = {
                'type': 'polyline', # Reuse the polyline logic for visualization
                'value': dist,
                'points': np.array([p_start, p_end]),
                'color': color
            }

    return results

# ==========================================
# 4. VISUALIZATION ENGINE
# ==========================================

def create_visualization(mesh, joints, measurements, idx_map, title="Body Analysis"):
    traces = []

    # Mesh
    x, y, z = mesh.vertices.T
    i, j, k = mesh.faces.T
    traces.append(go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, color='lightgray', opacity=0.3, name='Skin'))

    for name, data in measurements.items():
        color = data['color']
        val = data['value']

        if data['type'] == 'circ' and data['path'] is not None:
            traces.append(go.Scatter3d(
                x=data['path'][:, 0], y=data['path'][:, 1], z=data['path'][:, 2],
                mode='lines', line=dict(color=color, width=5), name=f"{name}"
            ))
        
        # Unified visualization for Polyline and Linear
        elif data['type'] == 'polyline':
            pts = data['points']
            # Draw the connected line
            traces.append(go.Scatter3d(
                x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                mode='lines+markers', line=dict(color=color, width=5), 
                marker=dict(size=4), name=f"{name}"
            ))
            
            # Floating Text Label (at the middle point of the chain)
            mid_idx = len(pts) // 2
            mid_pt = pts[mid_idx]
            
            # If it's a 2-point line, find exact geometric center
            if len(pts) == 2:
                mid_pt = np.mean(pts, axis=0)

            traces.append(go.Scatter3d(
                x=[mid_pt[0] + 5], y=[mid_pt[1]], z=[mid_pt[2] + 2],
                mode='text', text=[f"{val:.1f}cm"],
                textfont=dict(color=color, size=12, family="Arial Black"),
                showlegend=False
            ))

    # Joints
    rel_indices = [idx_map[k] for k in idx_map]
    rel_joints = joints[rel_indices]
    traces.append(go.Scatter3d(
        x=rel_joints[:, 0], y=rel_joints[:, 1], z=rel_joints[:, 2],
        mode='markers', marker=dict(size=4, color='red'), name='Joints'
    ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title, 
        scene=dict(aspectmode='data', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False),
        margin=dict(t=40, b=0, l=0, r=0)
    )
    return fig

# ==========================================
# 5. EXECUTION
# ==========================================
def get_measurements(mesh_orig, joints_orig, target_height_cm):
    print(f"--- ANALYZING (Target: {target_height_cm}cm) ---")
    mesh = mesh_orig.copy()
    joints = joints_orig.copy()
    mesh, joints, _ = scale_model(mesh, joints, target_height_cm)
    measures = calculate_body_metrics(mesh, joints, MHR_JOINT_MAP, MEASUREMENT_CONFIG)
    
    print("\n--- RESULTS ---")
    for k, v in measures.items():
        print(f"{k:<15}: {v['value']:.2f} cm")

    fig = create_visualization(mesh, joints, measures, MHR_JOINT_MAP, title=f"Curved Analysis")
    fig.show()

get_measurements(tpose_mesh, tpose_joints, 165)

--- ANALYZING (Target: 165cm) ---

--- RESULTS ---
Chest          : 110.86 cm
Waist          : 103.59 cm
Hips           : 115.50 cm
Spinal Length  : 50.73 cm
Arm Length     : 50.67 cm


### peek joints

In [8]:
import plotly.graph_objects as go
import numpy as np

def visualize_joints_with_indices(joints):
    print(f"Visualizing {len(joints)} joints with indices...")

    # Create list of index strings ("0", "1", "2"...)
    indices = [str(i) for i in range(len(joints))]

    # Create the 3D Scatter plot
    fig = go.Figure(data=[go.Scatter3d(
        x=joints[:, 0],
        y=joints[:, 1],
        z=joints[:, 2],
        mode='markers+text',       # Show both Dots and Numbers
        text=indices,              # The labels are the indices
        textposition="top center", # Put number slightly above the dot

        # Styling the Text
        textfont=dict(
            size=9,                # Keep small to reduce clutter
            color='black'
        ),

        # Styling the Dots
        marker=dict(
            size=5,
            color=np.arange(len(joints)), # Color gradient helps distinguish order
            colorscale='Jet',
            opacity=0.8
        ),

        # Hover Tooltip
        hoverinfo='text',
        hovertext=[f"Joint Index: {i}" for i in range(len(joints))]
    )])

    # Layout formatting
    fig.update_layout(
        title="Joint Index Finder (Scroll to Zoom, Drag to Rotate)",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data' # Keeps human proportions correct
        ),
        width=1000,
        height=800,
        margin=dict(r=0, l=0, b=0, t=40)
    )

    fig.show()

# --- RUN IT ---
# Use the joints from your generated T-Pose
if 'tpose_joints' in locals() and tpose_joints is not None:
    visualize_joints_with_indices(tpose_joints)
else:
    print("Error: 'tpose_joints' variable not found. Please run the T-Pose generator first.")

Visualizing 127 joints with indices...


## Pipeline

In [37]:
def pipe(model, file, height):
    img_bgr = cv2.imread(file)
    outputs = model.process_one_image(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

    try:
        # 1. Run the generator
        tpose_mesh, tpose_joints = get_tpose_mesh_final(estimator, outputs)
        print(f"\nSUCCESS: Generated T-Pose Mesh.")
        print(f"Vertices: {len(tpose_mesh.vertices)}")
        print(f"Joints: {len(tpose_joints) if tpose_joints is not None else 0}")

    except Exception as e:
        print(f"Error: {e}")
    
    get_measurements(tpose_mesh, tpose_joints, target_height_cm=height)

pipe(estimator, "/workspace/sam-3d-body-measurement/notebook/images/r.png", 163.0)

####### Please make sure the input image is in RGB format
Running object detector...
Found boxes: [[  42.64855   62.62031  562.77875 1103.5154 ]]
Running FOV estimator ...
Generating T-Pose with: Shape=45 + Pose=204 = 249

SUCCESS: Generated T-Pose Mesh.
Vertices: 18439
Joints: 127
--- ANALYZING (Target: 163.0cm) ---

--- RESULTS ---
Chest          : 94.86 cm
Waist          : 84.43 cm
Hips           : 104.23 cm
Spinal Length  : 50.06 cm
Arm Length     : 50.01 cm
